# ST-GCN on Gym99-skeleton

Gym99 is a subset of Gym288 — 99 fine-grained gymnastic action classes.
This notebook covers:
1. Repo setup
2. Joint inspection: all COCO-17 joints kept + virtual center (18 total)
3. Dataset download and tensor inspection
4. Training ST-GCN on Gym99
5. Inference / evaluation on the test split

## 1. Repo Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'duy'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', 'huggingface_hub', '-q'],
    check=True,
)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

## 2. Joint Inspection: COCO-17 + Virtual Center = 18 Joints

Gym99 annotations store keypoints in **COCO-17** format `(1, T, 17, 2)`.  
All 17 joints are kept (including face landmarks — eyes and ears contribute to the
centripetal/centrifugal adjacency partitions). A **virtual center** joint (index 17)
is appended as the mean of the four root joints (shoulders + hips).

Final ST-GCN input: `(N, C=2, T=64, J=18, M=1)`

In [ ]:
from src.config import COCO17_JOINT_NAMES, COCO17_BONES_18
from src.graph import Graph_COCO17_18Nodes
import torch

print('COCO-17 + virtual center (18 joints):')
for i, name in enumerate(COCO17_JOINT_NAMES):
    note = '  ← virtual (mean of joints 5,6,11,12)' if name == 'center*' else ''
    face = '  [face]' if i in [1, 2, 3, 4] else ''
    print(f'  [{i:2d}] {name:<18s}{face}{note}')

print(f'\nTotal bone pairs (18-joint graph): {len(COCO17_BONES_18)}')
for i, j in COCO17_BONES_18:
    print(f'  ({i:2d}, {j:2d})  {COCO17_JOINT_NAMES[i]} — {COCO17_JOINT_NAMES[j]}')

# Show adjacency tensor shape and partition sizes
g = Graph_COCO17_18Nodes()
A = g.A
print(f'\nAdjacency tensor A shape: {tuple(A.shape)}  (K=3 partitions, V=18, V=18)')
for k, name in enumerate(['self-loop / equidistant', 'centripetal', 'centrifugal']):
    nnz = int((A[k] > 0).sum())
    print(f'  A[{k}] — {name}: {nnz} non-zero entries')

## 3. Download Gym99-skeleton Dataset

In [ ]:
from huggingface_hub import snapshot_download

GYM99_DIR = Path('/content/Gym99-skeleton')
GYM99_REPOS = ['Lozumi/Gym99-skeleton', 'Lozumi/Gym99-skeletons']

if GYM99_DIR.exists() and any(GYM99_DIR.iterdir()):
    print('Gym99 dataset already exists — skip download.')
else:
    GYM99_DIR.mkdir(parents=True, exist_ok=True)
    last_err = None
    for repo_id in GYM99_REPOS:
        try:
            print(f'Trying: {repo_id}')
            snapshot_download(
                repo_id=repo_id,
                repo_type='dataset',
                local_dir=str(GYM99_DIR),
                local_dir_use_symlinks=False,
            )
            print(f'Download done: {repo_id}')
            last_err = None
            break
        except Exception as exc:
            last_err = exc
            print(f'Failed {repo_id}: {exc}')

    if last_err is not None and not any(GYM99_DIR.iterdir()):
        raise RuntimeError(
            'Cannot download Gym99 dataset automatically. '
            'Update repo_id list or place dataset manually at ' + str(GYM99_DIR)
        )

pkl_candidates = sorted(GYM99_DIR.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('No .pkl file found inside ' + str(GYM99_DIR))

GYM99_DATASET_PATH = pkl_candidates[0]
OUT_DIR = Path('outputs/gym99')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset pickle:', GYM99_DATASET_PATH)
print('Output dir    :', OUT_DIR)

print('\nDirectory contents:')
for item in sorted(GYM99_DIR.iterdir()):
    print(f'  {item.name}')

## 4. Inspect Dataset — Split Stats & Tensor Shapes

In [ ]:
import pickle
import numpy as np

with open(GYM99_DATASET_PATH, 'rb') as f:
    raw = pickle.load(f)

split_info   = raw.get('split', {})
annotations  = raw.get('annotations', [])
train_ids    = set(split_info.get('train', []))
test_ids     = set(split_info.get('test', []))

print(f'Total annotations : {len(annotations)}')
print(f'Train split IDs   : {len(train_ids)}')
print(f'Test split IDs    : {len(test_ids)}')

ann0 = annotations[0]
kp0  = np.asarray(ann0['keypoint'])
print(f'\nSample annotation keys : {list(ann0.keys())}')
print(f'Raw keypoint shape     : {kp0.shape}  (expected: 1, T, 17, 2 — COCO-17)')
print(f'Label                  : {ann0["label"]}')

labels_all = [int(a['label']) for a in annotations]
unique_labels = sorted(set(labels_all))
print(f'\nUnique labels : {len(unique_labels)}  (min={min(unique_labels)}, max={max(unique_labels)})')

In [ ]:
import torch
from src.gym99_dataset import build_gym99_data_tensors, infer_num_gym99_classes
from src.model import Model_STGCN_COCO18

# Load all splits
data, labels, flags, raw_counts, video_ids = build_gym99_data_tensors(
    dataset_path=str(GYM99_DATASET_PATH),
    split='all',
    keep_unknown_split=False,
)

train_mask = flags == 1
test_mask  = flags == 0

print(f'Total loaded  : {len(data)}')
print(f'Train samples : {train_mask.sum()}')
print(f'Test  samples : {test_mask.sum()}')
print(f'Tensor shape  : {data.shape}  (N, C=2, T=64, J=18, M=1)')
print(f'Labels shape  : {labels.shape}')
print(f'Dtype         : {data.dtype}')

num_classes = infer_num_gym99_classes(str(GYM99_DATASET_PATH))
print(f'\nInferred num_classes: {num_classes}')

# Quick forward pass (randomly initialized model)
n = min(4, len(data))
x = torch.from_numpy(data[:n]).float()
model = Model_STGCN_COCO18(num_classes=num_classes).eval()
with torch.no_grad():
    logits = model(x)
print(f'\nForward pass — input: {tuple(x.shape)}  output: {tuple(logits.shape)}')
print('Predictions (untrained):', torch.argmax(logits, dim=1).tolist())

## 5. Train ST-GCN on Gym99

Uncomment and run the cell below. Use `--use_two_stream` to enable the joint+bone two-stream variant.
Weights are saved as `stgcn_gym99_coco18.pth` (or `stgcn_gym99_coco18_2s.pth` for two-stream).

In [ ]:
# !python3 scripts/train_gym99.py \
#     --dataset_path /content/Gym99-skeleton/gym_99_skeleton.pkl \
#     --out_dir outputs/gym99 \
#     --epochs 30 \
#     --batch_size 64 \
#     --lr 0.001 \
#     --num_workers 2
#     # --use_two_stream

## 6. Inference / Evaluation on Test Split

After training completes, evaluate on the held-out test split.  
Update `--weights` to point to the saved `.pth` file.

In [ ]:
# !python3 scripts/inference_gym99.py \
#     --dataset_path /content/Gym99-skeleton/gym_99_skeleton.pkl \
#     --weights outputs/gym99/stgcn_gym99_coco18.pth \
#     --out_dir outputs/gym99 \
#     --batch_size 64 \
#     --topk 5
#     # --use_two_stream  # add if trained with two-stream